# CUDA Image Filter Analysis

This notebook analyzes the Caliper (`.cali`) output files from the CSCE 435 Lab 3 assignment using **Thicket** on the HPRC cluster.

**Requirements:**
1. `.cali` files must be in a folder named `cali_files` located in the same directory as this notebook.
2. This notebook is configured for the CSCE 435 HPRC environment.

In [ ]:
# HPRC Cluster Imports
import sys
# Add the specific site-packages path provided
sys.path.append("/scratch/group/csce-435-f25/python-3.10.8/lib/python3.10/site-packages")

from glob import glob
import matplotlib.pyplot as plt
import pandas as pd
import thicket as th
import math
import os

# Set visual style for plots
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load and Preprocess Data using Thicket

We use `thicket` to read all `.cali` files. Adiak values (like timers and problem sizes) are stored in the thicket metadata.

In [ ]:
def load_thicket_data(folder_path):
    if not os.path.exists(folder_path):
        print(f"Error: Directory '{folder_path}' not found.")
        return pd.DataFrame()

    cali_files = glob(os.path.join(folder_path, "*.cali"))
    
    if not cali_files:
        print("No .cali files found in the directory.")
        return pd.DataFrame()

    print(f"Found {len(cali_files)} files. Reading with Thicket...")
    
    try:
        # Load data into a Thicket object
        t = th.Thicket.from_caliper(cali_files)
        
        # Adiak values (timers, sizes) are stored in t.metadata
        # We make a copy to avoid modifying the original thicket structure if needed later
        df = t.metadata.copy()
        
        # Ensure the relevant columns exist and are numeric
        # These keys must match the adiak::value strings in your C++ code
        required_cols = [
            'image_size', 
            'filter_size',
            'cudaMemcpy_host_to_device_ms',
            'cudaMemcpy_device_to_host_ms',
            'kernel_global_time_ms',
            'kernel_constant_time_ms',
            'bandwidth_global_GBs',
            'bandwidth_constant_GBs'
        ]
        
        # Filter to keep only columns that actually exist in the data
        existing_cols = [c for c in required_cols if c in df.columns]
        
        if len(existing_cols) < len(required_cols):
            missing = set(required_cols) - set(existing_cols)
            print(f"Warning: The following expected columns were not found in the .cali files: {missing}")
        
        # Subset the dataframe
        df_clean = df[existing_cols].copy()
        
        # Convert to numeric (sometimes Thicket reads metadata as objects/strings)
        for col in existing_cols:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            
        return df_clean

    except Exception as e:
        print(f"Error reading files with Thicket: {e}")
        return pd.DataFrame()

# Load the data
df_raw = load_thicket_data('cali_files')

# Display first few rows if successful
if not df_raw.empty:
    print("Data loaded successfully.")
    display(df_raw.head())
else:
    print("Data load failed or empty.")

In [ ]:
# Group by configuration and find the MINIMUM time/MAXIMUM bandwidth
# Logic: Run multiple times, pick the fastest run (min time) to exclude OS jitter/outliers.

if not df_raw.empty:
    # Aggregate: Min for times, Max for bandwidths
    # We use a dictionary comprehension to only aggregate columns that exist
    agg_rules = {}
    
    if 'cudaMemcpy_host_to_device_ms' in df_raw.columns: agg_rules['cudaMemcpy_host_to_device_ms'] = 'min'
    if 'cudaMemcpy_device_to_host_ms' in df_raw.columns: agg_rules['cudaMemcpy_device_to_host_ms'] = 'min'
    if 'kernel_global_time_ms' in df_raw.columns:        agg_rules['kernel_global_time_ms'] = 'min'
    if 'kernel_constant_time_ms' in df_raw.columns:      agg_rules['kernel_constant_time_ms'] = 'min'
    if 'bandwidth_global_GBs' in df_raw.columns:         agg_rules['bandwidth_global_GBs'] = 'max'
    if 'bandwidth_constant_GBs' in df_raw.columns:       agg_rules['bandwidth_constant_GBs'] = 'max'
    
    # Group by image_size and filter_size
    # We dropna to ensure we don't have partial runs messing up the grouping
    df_agg = df_raw.dropna().groupby(['image_size', 'filter_size']).agg(agg_rules).reset_index()
    
    # Sort for cleaner plotting
    df_agg = df_agg.sort_values(by=['filter_size', 'image_size'])
    
    print("Aggregated Data (Best Runs):")
    display(df_agg)
else:
    df_agg = pd.DataFrame()

## 2. Visualization

Generating the 5 plots requested in the assignment.

In [ ]:
def plot_line_graph(df, x_col, y_cols, labels, title, ylabel, log_y=False):
    plt.figure(figsize=(10, 6))
    
    markers = ['o', 's', '^', 'D']
    
    for i, col in enumerate(y_cols):
        if col in df.columns:
            plt.plot(df[x_col], df[col], marker=markers[i % len(markers)], label=labels[i], linewidth=2)
        else:
            print(f"Warning: Column {col} not found in data, skipping line.")
    
    plt.title(title, fontsize=14)
    plt.xlabel('Image Size (NxN)', fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.legend(fontsize=12)
    
    # Ensure x-axis shows specific image sizes
    if not df.empty:
        plt.xticks(df[x_col].unique())
    
    if log_y:
        plt.yscale('log')
        
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

### Plot 1: Image Size vs. CudaMemCpy Time
Comparing Host-to-Device and Device-to-Host transfer times.

In [ ]:
if not df_agg.empty:
    # We only need one filter size for this, as memcpy is independent of filter computation
    # We'll take filter_size == 3 (or the first available)
    avail_filters = df_agg['filter_size'].unique()
    if len(avail_filters) > 0:
        # Use the smallest filter size available for this plot
        subset = df_agg[df_agg['filter_size'] == avail_filters[0]]
        
        plot_line_graph(
            subset, 
            'image_size', 
            ['cudaMemcpy_host_to_device_ms', 'cudaMemcpy_device_to_host_ms'],
            ['Host to Device', 'Device to Host'],
            'Image Size vs. CudaMemCpy Time',
            'Time (ms)'
        )

### Plots 2 & 3: Image Size vs. Kernel Time
Comparing Global Memory vs. Constant Memory implementations for 3x3 and 5x5 filters.

In [ ]:
if not df_agg.empty:
    filter_sizes = sorted(df_agg['filter_size'].unique())
    
    for f_size in filter_sizes:
        subset = df_agg[df_agg['filter_size'] == f_size]
        
        plot_line_graph(
            subset, 
            'image_size', 
            ['kernel_global_time_ms', 'kernel_constant_time_ms'],
            ['Global Memory Kernel', 'Constant Memory Kernel'],
            f'Image Size vs. Kernel Time ({int(f_size)}x{int(f_size)} Filter)',
            'Time (ms)'
        )

### Plots 4 & 5: Image Size vs. Effective Bandwidth
Comparing Global vs. Constant memory bandwidth throughput for 3x3 and 5x5 filters.

In [ ]:
if not df_agg.empty:
    filter_sizes = sorted(df_agg['filter_size'].unique())
    
    for f_size in filter_sizes:
        subset = df_agg[df_agg['filter_size'] == f_size]
        
        plot_line_graph(
            subset, 
            'image_size', 
            ['bandwidth_global_GBs', 'bandwidth_constant_GBs'],
            ['Global Memory', 'Constant Memory'],
            f'Image Size vs. Effective Bandwidth ({int(f_size)}x{int(f_size)} Filter)',
            'Bandwidth (GB/s)'
        )